In [1]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#Import AnimalShelter class
from CRUD import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Hardcode MongoDB username and password to allow for database access
username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(password, username)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))



#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add Grazioso Salvare's logo
image_filename = 'GraziosoSalvareLogo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())


app.layout = html.Div([
    # Title of Dashobard and Logo
    html.Center(children=[
        html.B(html.H1('SNHU CS-340 Dashboard')),
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'width': '200px'}),
        # Unique Identifier
        html.Center(html.H4('Nate Riggs CS-340 Dashboard')),
    ]),
    
        
# Interactive Filtering Options
    html.Hr(),
    html.Div([
        
        #  Filtering options for clients preferences for different use cases of dogs and their associated breeds
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water_rescue'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain_wilderness_rescue'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster_individual_tracking'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block'}
        )
    ]),
    html.Hr(),
    
    # Interactive Datatable with sorting, filtering, and pagination
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
        row_selectable='single',
        page_size = 10,
        sort_action='native',
        sort_mode='multi',                 
        filter_action='native',
        style_table={'height': '500px', 'overflowY': 'auto'},
        style_cell={'textAlign': 'center', 'minWidth': '50px', 'maxWidth': '180px', 'whiteSpace': 'normal'},
 ),
    html.Br(),
    html.Hr(),
    
#This sets up the dashboard so that the chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])

# Update dashboard depending on what the filter type is
def update_dashboard(filter_type):
    query = {}
    if filter_type == 'water_rescue':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'mountain_wilderness_rescue':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'disaster_individual_tracking':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    data = pd.DataFrame.from_records(db.read(query))
    return data.to_dict('records')


@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('filter-type', 'value')]
)
# Update graph to only show what is displayed in datatable
def update_graphs(viewData, filter_type):
    if viewData is None or len(viewData) == 0:
        return html.Div("No data available to display.")

    dff = pd.DataFrame.from_dict(viewData)
    
    # Reset filter is selected
    if filter_type == 'reset':
        # Count all different breed types
        breed_counts = dff['breed'].value_counts()
        
        # Top 5 breeds based on total count
        top_breeds = breed_counts.nlargest(5)
        
        # All other breeds combined into one part of chart for cleanliness
        other_count = breed_counts.iloc[5:].sum()

        pie_data = pd.concat([top_breeds, pd.Series({'Other': other_count})]).reset_index()
        pie_data.columns = ['breed', 'count']

        fig = px.pie(
            pie_data,
            names='breed',
            values='count',
            title='Top 5 Breeds (Grouped)',
            color_discrete_sequence=px.colors.qualitative.Set3

        )
    # If a filter is selected
    else:
        fig = px.pie(
            dff,
            names='breed',
            title='Preferred Animals by Breed',
            
            color_discrete_sequence=px.colors.qualitative.Set3
        )

    return [dcc.Graph(figure=fig)]


    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):    
    
    # if viewData is blank, display a default map
    if viewData is None or len(viewData) == 0:
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'},
                   center=[30.75, -97.48], zoom=10,
                   children=[
                       dl.TileLayer(id="base-layer-id"),
                       dl.Popup([html.P("No data to display.")])
                   ])
        ]
    dff = pd.DataFrame.from_dict(viewData)
    
    # If index is blank, display default map
    if index is None or len(index) == 0:
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'},
                   center=[30.75, -97.48], zoom=10,
                   children=[
                       dl.TileLayer(id="base-layer-id")
                   ])
        ]
    # If row is selected and data exists, show a marker
    row = index[0]
    
    # Try to execute using column names, will throw exception if unable to extract information from MongoDB
    try:
        lat = dff.iloc[row]['location_lat']
        lon = dff.iloc[row]['location_long']
        name = dff.iloc[row]['name']
        breed = dff.iloc[row]['breed']
    except Exception as e:
        print("Error extracting location:", e)
        return [html.P("Could not extract location data.")]
    
    # Return map with pin in location of animal, will display animal's name when selected
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[lat, lon], zoom=10,
               children=[
                   dl.TileLayer(id="base-layer-id"),
                   dl.Marker(position=[lat, lon], children=[
                       dl.Tooltip(f"Breed: {breed}"),
                       dl.Popup([
                           html.H1("Animal Name", style={'fontSize': '15px', 'textDecoration': 'underline'}),
                           html.P(name, style={'fontSize': '15px'})
                       ])
                   ])
               ])
    ]


app.run_server(debug=True)


Dash app running on http://127.0.0.1:13146/
